<a href="https://colab.research.google.com/github/GGSimmons1992/UTYV6k8pXAuLL0yL/blob/main/Notebooks/ClassicalRandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Classical Random Forest

In [ ]:
import numpy
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as rf
import pickle
from imblearn.over_sampling import SMOTENC
from os.path import exists
from sklearn.preprocessing import StandardScaler
import category_encoders as ce
from sklearn.feature_selection import chi2
from scipy.stats import spearmanr
import json
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
def fileExistsInDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  return exists(fullName)

In [ ]:
def savePickleFileToDrive(data,filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  pickle.dump(data, open(fullName, 'wb'))

In [ ]:
def retreivePickleFileFromDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  return pickle.load(open(fullName, 'rb'))

In [ ]:
def saveJSONToDrive(data,filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  with open(fullName, 'w') as fp:
      json.dump(data, fp)

In [ ]:
def retrieveJSONFromDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  with open(fullName) as json_file:
      data = json.load(json_file)
  return data

In [ ]:
def saveCSVToDrive(df,filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  df.to_csv(fullName,index=False)

In [ ]:
def retrieveCSVFromDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/SalesReinforcerData/{filename}"
  return pd.read_csv(fullName)

In [ ]:
def separateDFBySubtype(df,baseName):
  numericalCols = []
  categoricalCols = []
  for col in df.columns:
      if np.issubdtype(df[col].dtype, np.number):
          numericalCols.append(str(col))
      else:
          categoricalCols.append(str(col))
  numericalDF = df[numericalCols]
  categoricalDF = df[categoricalCols]
  pickle.dump(numericalCols, open(f"../Data/Interim/{baseName}NumericalCols.pkl", 'wb'))
  pickle.dump(categoricalCols, open(f"../Data/Interim/{baseName}CategoricalCols.pkl", 'wb'))
  return numericalDF,categoricalDF

In [ ]:
def removeUnimportantCategoricalColumns(categoricalDF,y,combinedName,datasetType = 'train'):
  if datasetType == 'test':
      significantColumns = pickle.load(open(f"../Data/Interim/{combinedName}SignificantCategoricalCols.pkl", 'rb'))
  else:
      le = ce.OrdinalEncoder(return_df=True)
      leDF = le.fit_transform(categoricalDF)
      pValues = chi2(leDF, y)[1]
      pValueDF = pd.DataFrame({"feature":list(categoricalDF.columns),"pValue":pValues},columns=["feature","pValue"],index=None)
      lowPDF = pValueDF[pValueDF["pValue"] < 0.05]
      significantColumns = list(lowPDF['feature'])
      pickle.dump(significantColumns, open(f"../Data/Interim/{combinedName}SignificantCategoricalCols.pkl", 'wb'))
  return categoricalDF[significantColumns]

In [ ]:
def removeUnimportantNumericalColumns(numericalDF,y,combinedName,datasetType = 'train'):
  if datasetType == 'test':
      significantColumns = pickle.load(open(f"../Data/Interim/{combinedName}SignificantNumericalCols.pkl", 'rb'))
  else:
      numericalCols = list(numericalDF.columns)
      significantColumns = []
      allNumericalColumns = numericalDF.columns
      for col in allNumericalColumns:
          x = numericalDF[[col]].values.ravel()
          p = spearmanr(x,y)[1]
          if p < 0.05:
              significantColumns.append(str(col))
      pickle.dump(significantColumns, open(f"../Data/Interim/{combinedName}SignificantNumericalCols.pkl", 'wb'))
  return numericalDF[significantColumns]

In [ ]:
def encodeTestDF(categoricalDF,baseName):
  ohe = retrievePickleFileFromDrive(f"{baseName}OneHotEncoder.pkl")
  le = retrievePickleFileFromDrive(f"{baseName}LabelEncoder.pkl")
  oheDF = ohe.transform(categoricalDF).fillna(0)
  leDF = le.transform(categoricalDF)
  return pd.concat([oheDF,leDF],axis=1)

In [ ]:
def encodeDF(categoricalDF,baseName):
  ohe = ce.OneHotEncoder(handle_unknown='ignore',return_df=True,use_cat_names=True)
  le = ce.OrdinalEncoder(return_df=True)
  oheDF = ohe.fit_transform(categoricalDF)
  oheColumns = list(oheDF.columns)
  savePickleFileToDrive(oheColumns,f"{baseName}OheColumns.pkl")
  leDF = le.fit_transform(categoricalDF)
  savePickleFileToDrive(ohe,f"{baseName}OneHotEncoder.pkl")
  savePickleFileToDrive(le,f"{baseName}LabelEncoder.pkl")
  return pd.concat([oheDF,leDF],axis=1)

In [ ]:
def scaleTestDF(df,baseName):
  scaler = retreivePickleFileFromDrive(f"{baseName}Scaler.pkl")
  numericalCols = list(df.columns)
  df[numericalCols] = scaler.transform(df[numericalCols])
  return df

In [ ]:
def scaleDF(df,baseName):
  scaler = StandardScaler()
  numericalCols = list(df.columns)
  df[numericalCols] = scaler.fit_transform(df[numericalCols])
  savePickleFileToDrive(scaler,f"{baseName}Scaler.pkl")
  return df

In [ ]:
def processTestData(baseName):
  combinedName = baseName + balanceType
  df = retrieveCSVFromDrive(f"{combinedName}Test.csv")
  yArray = df[['isSubscribed']].values.ravel()
  df = df.drop('isSubscribed',axis = 1)
  numericalDF = removeUnimportantNumericalColumns(df,yArray,combinedName,"test")
  categoricalDF = removeUnimportantCategoricalColumns(df,yArray,combinedName,"test")
  scaledDF = scaleTestDF(numericalDF,combinedName)
  encodedDF = encodeTestDF(categoricalDF,combinedName)
  finalDF = pd.concat([scaledDF,encodedDF],axis=1)
  finalDF['isSubscribed'] = yArray.reshape(-1,1)
  saveCSVToDrive(finalDF,f"{combinedName}Test.csv")

In [ ]:
def processTrainData(baseName):
  combinedName = baseName + balanceType
  df = pd.read_csv(f'../Data/Interim/{combinedName}Train.csv')
  yArray = df[['isSubscribed']].values.ravel()
  df = df.drop('isSubscribed',axis = 1)
  numericalDF,categoricalDF = separateDFBySubtype(df,baseName)
  numericalDF = removeUnimportantNumericalColumns(numericalDF,yArray,combinedName)
  categoricalDF = removeUnimportantCategoricalColumns(categoricalDF,yArray,combinedName)
  scaledDF = scaleDF(numericalDF,combinedName)
  encodedDF = encodeDF(categoricalDF,combinedName)
  finalDF = pd.concat([scaledDF,encodedDF],axis=1)
  finalDF['isSubscribed'] = yArray.reshape(-1,1)
  saveCSVToDrive(finalDF,f"{combinedName}Train.csv")


In [ ]:
def transformData(data):
  dateTimeColumns = ['First Contact','Last Contact', 'First Call', 'Signed up for a demo',
                     'Filled in customer survey','Did sign up to the platform','Account Manager assigned','Subscribed']
  for col in dateTimeColumns:
    data[col] = pd.to_datetime(data[col])
    data[col + 'Year'] = data[col].dt.year
    data[col + 'Month'] = data[col].dt.month
    data[col + 'Day'] = data[col].dt.day
    data[col] = data[col].astype('int64')
  return data

In [ ]:
def binarizeTargetsFromDF(df):
    df['isSubscribed'] = df['Subscribed'].notna().astype(int)
    return df

In [ ]:
def smoteThenSplit(X,y,baseName):
    categoricalVariables = [i for i,col in enumerate(X.columns) if not np.issubdtype(X[col].dtype, np.number)]
    balancer = SMOTENC(categorical_features=categoricalVariables,random_state=51)
    categoricalVariables = [col for col in X.columns if not np.issubdtype(X[col].dtype, np.number)]
    encoder = ce.OneHotEncoder(cols=categoricalVariables)
    X_encoded = encoder.fit_transform(X)
    resampledX,resampledy = balancer.fit_resample(X_encoded, y)
    return encoder.inverse_transform(resampledX),resampledy

In [ ]:
def splitData(baseName):
    df = retrieveCSVFromDrive("dateTransformedData.csv")
    df = binarizeTargetsFromDF(df)
    originalY = df[['isSubscribed']]
    originalYArray = originalY.values.ravel()
    originalX = df.drop('isSubscribed',axis=1)
    smoteThenSplit(originalX,originalYArray,baseName)

In [ ]:
def main():
  data = retreiveCSVFromDrive("SalesCRM - CRM.csv")
  dateTransformedData = transformDate(data)
  saveCSVToDrive(dateTransformedData,"dateTransformedData.csv")
  np.random.seed(51)
  baseName = "SalesReinforcer"
  if (fileExistsInDrive(f"{baseName}Train.csv") == False):
      splitData(baseName)
  processTrainData(baseName)
  processTestData(baseName)


In [ ]:
if __name__ == "__main__":
    main()

Original datetime: 2023-01-01 12:00:00+00:00
Converted to int64 (nanoseconds since epoch): 1672574400000000000
Converted to seconds since epoch: 1672574400.0
